In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
import time

# 1. 加载数据
data = pd.read_csv('total_load_actual.csv')

# 2. 检查数据是否包含 NaN
if data.isnull().any().any():
    print("Data contains NaN values. Filling missing values...")
    data = data.ffill()  # 使用前值填充缺失数据

# 3. 将数据列转换为字典格式
data_dict = {}
for col in data.columns:
    data_dict[col] = data[col].tolist()

total_acual_load_list = data_dict['total load actual']

# 4. 归一化数据
scaler = MinMaxScaler(feature_range=(0, 1))
total_acual_load_list_scaled = scaler.fit_transform(np.array(total_acual_load_list).reshape(-1, 1)).flatten()

# 5. 设定序列长度和样本数
seq_length = 24
num_samples = len(total_acual_load_list_scaled) - seq_length
# num_samples = 1000 # 为了快速测试，只使用1000个样本
# 6. 初始化输入和输出数据
x_processed = []
y_processed = []

window_size = 2
i = 0
while i * window_size + seq_length-1 < len(total_acual_load_list_scaled):
    x_processed.append(total_acual_load_list_scaled[i * window_size:i * window_size + seq_length-2])
    y_processed.append(total_acual_load_list_scaled[i * window_size + seq_length-1])
    i += 1

# 7. 转换为NumPy数组
x_processed = np.array(x_processed)
y_processed = np.array(y_processed)

# 8. 转换为Tensor
x_processed_tensor = torch.tensor(x_processed, dtype=torch.float32)
y_processed_tensor = torch.tensor(y_processed, dtype=torch.float32).reshape(-1, 1)

# 9. 数据形状检查：确保 x_processed_tensor 的形状为 (样本数量, 序列长度-1, 输入维度)
x_processed_tensor = x_processed_tensor.unsqueeze(-1)  # 将输入扩展为三维张量 (样本数量, seq_length-1, input_dim)

# 10. 数据划分：80% 训练集，20% 测试集 使用 train_test_split
x_train, x_test, y_train, y_test = train_test_split(x_processed_tensor, y_processed_tensor, test_size=0.2, random_state=42)

# 11. 定义改进的堆叠GRU-RNN模型
class ImprovedGRUCell(nn.Module):
    def __init__(self, input_dim, hidden_dim):
        super(ImprovedGRUCell, self).__init__()
        self.hidden_dim = hidden_dim
        
        # 更新门和重置门的权重和偏置
        self.update_gate = nn.Linear(hidden_dim, hidden_dim, bias=True)
        self.reset_gate = nn.Linear(hidden_dim, hidden_dim, bias=True)
        
        # 候选层的权重和偏置
        self.candidate_layer = nn.Linear(input_dim + hidden_dim, hidden_dim, bias=True)
        
    def forward(self, x, h):
        # 计算更新门
        z = torch.sigmoid(self.update_gate(h))
        
        # 计算重置门
        r = torch.sigmoid(self.reset_gate(h))
        
        # 计算候选层
        h_tilde = torch.tanh(self.candidate_layer(torch.cat([x, r * h], dim=1)))
        
        # 更新隐藏状态
        h_new = (1 - z) * h + z * h_tilde
        
        return h_new

class StackedGRU(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, num_layers=2):
        super(StackedGRU, self).__init__()
        self.hidden_dim = hidden_dim
        self.num_layers = num_layers
        
        # 使用改进的GRU单元
        self.gru_cells = nn.ModuleList([ImprovedGRUCell(input_dim, hidden_dim)])
        for _ in range(1, num_layers):
            self.gru_cells.append(ImprovedGRUCell(hidden_dim, hidden_dim))
        
        self.fc = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        h = [torch.zeros(x.size(0), self.hidden_dim).to(x.device) for _ in range(self.num_layers)]
        
        for t in range(x.size(1)):
            for layer in range(self.num_layers):
                if layer == 0:
                    h[layer] = self.gru_cells[layer](x[:, t, :], h[layer])
                else:
                    h[layer] = self.gru_cells[layer](h[layer-1], h[layer])
        
        out = self.fc(h[-1])
        return out

# 12. 定义模型
input_dim = 1  # 输入维度
hidden_dim = 32
output_dim = 1
num_layers = 1
model = StackedGRU(input_dim, hidden_dim, output_dim, num_layers)

# 检查是否有可用的GPU，如果有则使用GPU，否则使用CPU
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

# 13. 定义损失函数和优化器
criterion = nn.MSELoss()
# optimizer = optim.Adagrad(
#     model.parameters(), 
#     lr=0.1, 
#     lr_decay=0.0, 
#     weight_decay=1e-5, 
#     initial_accumulator_value=0.0, 
#     eps=1e-10
# )

# 使用adams
optimizer = optim.Adam(model.parameters(), lr=0.05)
# 14. 训练模型
num_epochs = 1000
train_loss = []
print('Training model...')
start_time = time.time()  # 记录训练开始时间

for epoch in range(num_epochs):
    model.train()
    optimizer.zero_grad()
    
    # 将数据移动到设备上
    x_train_device = x_train.to(device)
    y_train_device = y_train.to(device)
    
    output = model(x_train_device)
    loss = criterion(output, y_train_device)
    loss.backward()
    optimizer.step()
    
    train_loss.append(loss.item())
    if (epoch+1) % (num_epochs//10) == 0:
        print(f'Epoch [{epoch+1}/{num_epochs}], Loss: {loss.item():.4f}')

end_time = time.time()  # 记录训练结束时间
training_time = end_time - start_time  # 计算训练耗时
print(f'Training completed in {training_time:.2f} seconds')

Data contains NaN values. Filling missing values...
Training model...
Epoch [100/1000], Loss: 0.0046
Epoch [200/1000], Loss: 0.0040
Epoch [300/1000], Loss: 0.0027
Epoch [400/1000], Loss: 0.0027


In [ ]:
from sklearn.metrics import mean_squared_error
import numpy as np

# 计算各类指标的函数
def calculate_metrics(y_true, y_pred):
    mse = mean_squared_error(y_true, y_pred)
    mae = np.mean(np.abs(y_true - y_pred))
    mape = np.mean(np.abs((y_true - y_pred) / y_true)) * 100
    rmse = np.sqrt(mse)
    r2 = 1 - (np.sum((y_true - y_pred) ** 2) / np.sum((y_true - np.mean(y_true)) ** 2))
    smape = np.mean(2 * np.abs(y_true - y_pred) / (np.abs(y_true) + np.abs(y_pred)))
    
    return mse, mae, mape, rmse, r2, smape

# 15. 测试模型
model.eval()
with torch.no_grad():
    predicted = model(x_test).cpu().numpy()

# 16. 反归一化预测结果
predicted_original_scale = scaler.inverse_transform(predicted)
y_test_original_scale = scaler.inverse_transform(y_test.cpu().numpy())

# 17. 计算归一化前的指标
mse_original, mae_original, mape_original, rmse_original, r2_original, smape_original = calculate_metrics(y_test_original_scale, predicted_original_scale)

# 18. 计算归一化后的指标
y_test_np = y_test.cpu().numpy().flatten()
predicted_np = predicted.flatten()
mse_normalized, mae_normalized, mape_normalized, rmse_normalized, r2_normalized, smape_normalized = calculate_metrics(y_test_np, predicted_np)

# 19. 打印结果
print('+=============+=============+=============+=============+')
print('|   Metrics (Original Scale)   |   Value   |   Metrics (Normalized Scale)   |   Value   |')
print('+=============+=============+=============+=============+')
metrics = [
    ('MSE', mse_original, mse_normalized),
    ('MAE', mae_original, mae_normalized),
    ('MAPE', mape_original, mape_normalized),
    ('RMSE', rmse_original, rmse_normalized),
    ('R²', r2_original, r2_normalized),
    ('SMAPE', smape_original, smape_normalized)
]

for metric, original_value, normalized_value in metrics:
    print(f'| {metric:<15} | {original_value:>26.6f} | {metric} (Normalized){"":<2} | {normalized_value:>26.6f} |')

print('+=============+=============+=============+=============+')


+=============+=============+=============+=============+
|   Metrics (Original Scale)   |   Value   |   Metrics (Normalized Scale)   |   Value   |
+=============+=============+=============+=============+
| MSE             |             1605238.125000 | MSE (Normalized)   |                   0.003041 |
| MAE             |                 899.088501 | MAE (Normalized)   |                   0.039135 |
| MAPE            |                   3.227895 | MAPE (Normalized)   |                  11.542725 |
| RMSE            |                1266.979980 | RMSE (Normalized)   |                   0.055148 |
| R²              |                   0.921512 | R² (Normalized)   |                   0.921512 |
| SMAPE           |                   0.032332 | SMAPE (Normalized)   |                   0.108239 |
+=============+=============+=============+=============+


In [ ]:
import plotly.graph_objs as go
import plotly.offline as pyo

# 1. 训练损失的交互式可视化
train_loss_fig = go.Figure()

# 添加训练损失曲线
train_loss_fig.add_trace(go.Scatter(
    x=list(range(1, num_epochs + 1)),  # 横坐标是训练的轮数
    y=train_loss,  # 纵坐标是训练过程中的损失
    mode='lines',  # 线形显示
    name='Training Loss',  # 曲线的标签
    line=dict(color='royalblue', width=3, dash='dash')  # 设置颜色、宽度和虚线样式
))

# 更新图表的布局
train_loss_fig.update_layout(
    title='TEST:Training Loss Curve',  # 图表标题
    title_font=dict(size=20, family='Arial', color='black'),  # 设置标题字体
    xaxis=dict(title='Epoch', title_font=dict(size=14, family='Arial', color='black'), gridcolor='lightgray'),  # X轴标签
    yaxis=dict(title='Loss', title_font=dict(size=14, family='Arial', color='black'), gridcolor='lightgray'),  # Y轴标签
    plot_bgcolor='whitesmoke',  # 设置图表背景色
    paper_bgcolor='whitesmoke',  # 设置图表外部背景色
    font=dict(family='Arial', size=12, color='black'),  # 字体设置
    legend=dict(
        x=0.5, y=1.05,  # 设置图例位置
        xanchor='center',  # 图例水平居中
        yanchor='bottom',  # 图例垂直居上
        traceorder='normal',  # 图例顺序
        orientation='h',  # 图例显示为水平排列（即一行多列）
        font=dict(family='Arial', size=12, color='black'),
        bgcolor='rgba(255, 255, 255, 0.5)',  # 图例背景颜色
        bordercolor='black', borderwidth=1  # 图例边框颜色和宽度
    ),
    showlegend=True,  # 显示图例
    width=1200,  # 设置图表宽度
    height=600  # 设置图表高度
)

# 设置图表边缘的阴影效果
train_loss_fig.update_layout(
    xaxis=dict(showgrid=True, zeroline=False, showline=True, linecolor='gray'),  # X轴样式
    yaxis=dict(showgrid=True, zeroline=False, showline=True, linecolor='gray'),  # Y轴样式
    margin=dict(l=50, r=50, t=50, b=50)  # 设置图表的边距
)

train_loss_fig.show()

# 2. 预测与实际结果的交互式可视化
# 创建实际结果的曲线
timestep = 100
actual_trace = go.Scatter(
    x=list(range(timestep)),  # 横坐标是时间步
    y=y_test_original_scale[:timestep].flatten(),  # 纵坐标是实际测试数据（前100个数据点）
    mode='lines+markers',  # 线条和数据点显示
    name='Actual',  # 曲线名称
    line=dict(color='mediumseagreen', width=3, shape='spline'),  # 设置颜色、宽度和曲线类型
    marker=dict(symbol='circle', size=8, color='white', line=dict(color='mediumseagreen', width=2))  # 数据点样式（空心）
)

# 创建预测结果的曲线
predicted_trace = go.Scatter(
    x=list(range(timestep)),  # 横坐标是时间步
    y=predicted_original_scale[:timestep].flatten(),  # 纵坐标是模型的预测数据（前100个数据点）
    mode='lines+markers',  # 线条和数据点显示
    name='Predicted',  # 曲线名称
    line=dict(color='indianred', width=3, dash='dot'),  # 设置颜色、宽度和虚线样式
    marker=dict(symbol='x', size=8, color='white', line=dict(color='indianred', width=2))  # 数据点样式（空心）
)

# 创建包含实际和预测数据的图表
comparison_fig = go.Figure(data=[actual_trace, predicted_trace])

# 更新布局设置
comparison_fig.update_layout(
    title='TEST:Actual vs Predicted',  # 图表标题
    title_font=dict(size=20, family='Arial', color='black'),  # 标题字体
    xaxis=dict(title='Time Step', title_font=dict(size=14, family='Arial', color='black'), gridcolor='lightgray'),  # X轴标签
    yaxis=dict(title='Load', title_font=dict(size=14, family='Arial', color='black'), gridcolor='lightgray'),  # Y轴标签
    plot_bgcolor='whitesmoke',  # 图表背景颜色
    paper_bgcolor='whitesmoke',  # 外部背景颜色
    font=dict(family='Arial', size=12, color='black'),  # 字体设置
    legend=dict(
        x=0.5, y=1.05,  # 设置图例位置
        xanchor='center',  # 图例水平居中
        yanchor='bottom',  # 图例垂直居上
        traceorder='normal',  # 图例顺序
        orientation='h',  # 图例显示为水平排列（即一行多列）
        font=dict(family='Arial', size=12, color='black'),
        bgcolor='rgba(255, 255, 255, 0.5)',  # 图例背景颜色
        bordercolor='black', borderwidth=1  # 图例边框颜色和宽度
    ),
    showlegend=True,  # 显示图例
    width=1200,  # 设置图表宽度
    height=600  # 设置图表高度
)

# 为图表添加渐变背景色
comparison_fig.update_layout(
    paper_bgcolor='rgba(240, 240, 240, 0.9)',  # 外部背景渐变色
    plot_bgcolor='rgba(255, 255, 255, 0.9)',  # 图表区域背景渐变色
)

# 显示预测与实际结果的图表
comparison_fig.show()

### 改进堆叠GRU-RNN模型结构

#### 1. 改进的GRU单元

每个改进的GRU单元由以下部分组成：

- **更新门（Update Gate）**：
  $$
  \mathbf{z}_t = \sigma(\mathbf{U}_{hz} \mathbf{h}_{t-1} + \mathbf{b}_z)
  $$
  其中：
  - $\mathbf{z}_t$ 是更新门。
  - $\mathbf{U}_{hz}$ 是更新门的权重矩阵。
  - $\mathbf{h}_{t-1}$ 是前一个时间步的隐藏状态。
  - $\mathbf{b}_z$ 是更新门的偏置。
  - $\sigma$ 是sigmoid激活函数。

- **重置门（Reset Gate）**：
  $$
  \mathbf{r}_t = \sigma(\mathbf{U}_{hr} \mathbf{h}_{t-1} + \mathbf{b}_r)
  $$
  其中：
  - $\mathbf{r}_t$ 是重置门。
  - $\mathbf{U}_{hr}$ 是重置门的权重矩阵。
  - $\mathbf{h}_{t-1}$ 是前一个时间步的隐藏状态。
  - $\mathbf{b}_r$ 是重置门的偏置。
  - $\sigma$ 是sigmoid激活函数。

- **候选隐藏状态（Candidate Hidden State）**：
  $$
  \mathbf{c}_t = \varphi(\mathbf{V}_{xc} \mathbf{x}_t + \mathbf{U}_{hc} (\mathbf{r}_t \odot \mathbf{h}_{t-1}) + \mathbf{b}_c)
  $$
  其中：
  - $\mathbf{c}_t$ 是候选隐藏状态。
  - $\mathbf{V}_{xc}$ 是输入到候选隐藏状态的权重矩阵。
  - $\mathbf{x}_t$ 是当前时间步的输入。
  - $\mathbf{U}_{hc}$ 是隐藏状态到候选隐藏状态的权重矩阵。
  - $\mathbf{r}_t \odot \mathbf{h}_{t-1}$ 是重置门和前一个隐藏状态的逐元素乘积。
  - $\mathbf{b}_c$ 是候选隐藏状态的偏置。
  - $\varphi$ 是tanh激活函数。

- **新的隐藏状态（New Hidden State）**：
  $$
  \mathbf{h}_t = (1 - \mathbf{z}_t) \odot \mathbf{h}_{t-1} + \mathbf{z}_t \odot \mathbf{c}_t
  $$
  其中：
  - $\mathbf{h}_t$ 是新的隐藏状态。
  - $\mathbf{z}_t$ 是更新门。
  - $\mathbf{h}_{t-1}$ 是前一个时间步的隐藏状态。
  - $\mathbf{c}_t$ 是候选隐藏状态。
  - $\odot$ 表示逐元素乘法。

#### 2. 堆叠GRU-RNN模型

堆叠GRU-RNN模型由多个改进的GRU单元和一个回归层组成：

- **输入层**：
  - 输入维度：$\text{input\_dim}$

- **隐藏层**：
  - 隐藏层维度：$ \text{hidden\_dim} $
  - 层数：$ \text{num\_layers} $
  - 每个隐藏层由一个改进的GRU单元组成。

- **回归层**：
  - 输入维度：$ \text{hidden\_dim} $
  - 输出维度：$ \text{output\_dim} $
  - 回归层的计算：
    $$
    \hat{\mathbf{y}}_t = \sigma(\mathbf{U}_{yh} \mathbf{h}_t + \mathbf{b}_y)
    $$
    其中：
    - $\hat{\mathbf{y}}_t$ 是预测输出。
    - $\mathbf{U}_{yh}$ 是回归层的权重矩阵。
    - $\mathbf{h}_t$ 是最后一个GRU单元的隐藏状态。
    - $\mathbf{b}_y$ 是回归层的偏置。
    - $\sigma$ 是sigmoid激活函数。

### 训练算法

#### 1. 基本训练算法

使用带有动量的随机梯度下降（SGD）算法来调整权重和偏置，以最小化均方误差（MSE）：

$$
E = \sum_{t=1}^{T} (\mathbf{y}_t - \hat{\mathbf{y}}_t)^2 / 2T
$$

$$
\theta_{q+1} = \theta_q - \eta \cdot \left(\frac{\partial E}{\partial \theta_q}\right) + \beta \cdot (\theta_q - \theta_{q-1})
$$
其中：
- $E$ 是均方误差。
- $\theta_q$ 是第 $q$ 次迭代的参数集。
- $\mathbf{y}_t$ 是实际输出。
- $T$ 是训练样本总数。
- $\eta$ 是学习率。
- $\beta$ 是动量因子。

#### 2. 改进的训练算法

结合AdaGrad和可调动量的改进训练算法：

$$
\theta_{q+1} = \theta_q - \eta_q \cdot \left(\frac{\partial E_q}{\partial \theta_q}\right) + \beta_q \cdot (\theta_q - \theta_{q-1})
$$

$$
\eta_q = \frac{\eta_0}{\sqrt{\sum_{i=1}^{q} g_i^2 + \lambda}}
$$

$$
 g_q = \frac{\partial E_q}{\partial \theta_q}
$$

$$
\beta_q = e^{-\kappa - \|g_q\|}
$$

其中：
- $\eta_q$ 是可调学习率。
- $\beta_q$ 是可调动量因子。
- $\lambda$ 是一个小正数（默认1e-8）。
- $\kappa$ 是可调动量项的初始系数。
- $\eta_0$ 是初始学习率。

### 总结

论文中的神经网络模型是一个改进的堆叠GRU-RNN，通过减少模型参数和改进训练算法来提高预测性能。模型结构包括多层改进的GRU单元和一个回归层，训练算法结合了AdaGrad和可调动量，以适应不同的梯度变化。
